In [10]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("../data/raw/room_booking_requests.csv")
df

,booking_id,employee_id,booking_date,requested_start_datetime,requested_end_datetime,requested_duration_minutes,requested_room_type,requested_capacity,max_time_flexibility_minutes,requested_floor
0,RB06430,E0742,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Meeting,16,60,3
1,RB10999,E0166,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Conference,4,60,5
2,RB19847,E0605,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Conference,12,0,1
3,RB22582,E0904,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Meeting,4,30,3
4,RB25100,E0378,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:00:00,60,Meeting,10,0,5
...,...,...,...,...,...,...,...,...,...,...
39995,RB36542,E0686,2025-12-31,2025-12-31 10:45:00,2025-12-31 12:00:00,75,Meeting,12,15,5
39996,RB29461,E0138,2025-12-31,2025-12-31 11:00:00,2025-12-31 12:00:00,60,Conference,12,15,1
39997,RB24451,E0138,2025-12-31,2025-12-31 11:15:00,2025-12-31 12:15:00,60,Conference,12,15,1
39998,RB34216,E0144,2025-12-31,2025-12-31 11:15:00,2025-12-31 12:45:00,90,Training,10,60,1


In [ ]:
# 1000 unique employees
df["employee_id"].nunique()

1000

In [ ]:
df["employee_id"].value_counts().describe()

count    1000.000000
mean       40.000000
std        19.627155
min         2.000000
25%        32.000000
50%        38.000000
75%        50.000000
max        91.000000
Name: count, dtype: float64

In [ ]:
#=============================================== EMPLOYEE LEVEL REQUEST DISTRIBUTION ========================================================
employee_requests = (
    df.groupby("employee_id")
      .size()
      .reset_index(name="request_count")
      .sort_values("request_count", ascending=False)
)

employee_requests.head(10)

,employee_id,request_count
958,E0959,91
299,E0300,89
764,E0765,88
110,E0111,86
45,E0046,86
616,E0617,86
869,E0870,84
793,E0794,83
483,E0484,83
393,E0394,82


In [ ]:
#============================================== Full distribution of employees by request count =============================================
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

labels = [
    "1-10", "11-20", "21-30", "31-40", "41-50",
    "51-60", "61-70", "71-80", "81-90", "91-100"
]

employee_requests["request_range"] = pd.cut(
    employee_requests["request_count"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

employee_requests["request_range"].value_counts().sort_index()

request_range
1-10      124
11-20      32
21-30      66
31-40     382
41-50     148
51-60      52
61-70     114
71-80      66
81-90      15
91-100      1
Name: count, dtype: int64

In [ ]:
df["booking_date"] = pd.to_datetime(df["booking_date"])

In [ ]:
monthly_activity = (
    df.groupby(df["booking_date"].dt.to_period("M"))
      .agg(
          request_count=("booking_id", "count"),
          unique_employees=("employee_id", "nunique")
      )
      .reset_index()
)

monthly_activity

,booking_date,request_count,unique_employees
0,2025-01,3475,903
1,2025-02,3008,869
2,2025-03,3228,894
3,2025-04,3385,897
4,2025-05,3261,891
5,2025-06,3250,891
6,2025-07,3603,895
7,2025-08,3166,890
8,2025-09,3412,904
9,2025-10,3623,905


In [ ]:
monthly_activity["requests_per_employee"] = (
    monthly_activity["request_count"] /
    monthly_activity["unique_employees"]
)

monthly_activity

,booking_date,request_count,unique_employees,requests_per_employee
0,2025-01,3475,903,3.848283
1,2025-02,3008,869,3.461450
2,2025-03,3228,894,3.610738
3,2025-04,3385,897,3.773690
4,2025-05,3261,891,3.659933
5,2025-06,3250,891,3.647587
6,2025-07,3603,895,4.025698
7,2025-08,3166,890,3.557303
8,2025-09,3412,904,3.774336
9,2025-10,3623,905,4.003315


In [ ]:
room_type_behavior = (
    df.groupby(["employee_id", "requested_room_type"])
      .size()
      .reset_index(name="request_count")
)

room_type_behavior

,employee_id,requested_room_type,request_count
0,E0001,Focus,5
1,E0001,Meeting,58
2,E0001,Training,2
3,E0002,Conference,4
4,E0002,Meeting,6
...,...,...,...
3153,E0999,Meeting,37
3154,E0999,Training,3
3155,E1000,Conference,1
3156,E1000,Meeting,1


In [ ]:
room_type_behavior["percentage"] = (
    room_type_behavior["request_count"]
    / room_type_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

In [ ]:
room_type_behavior[room_type_behavior["employee_id"] == "E0298"]

,employee_id,requested_room_type,request_count,percentage
933,E0298,Conference,2,5.405405
934,E0298,Focus,1,2.702703
935,E0298,Meeting,32,86.486486
936,E0298,Training,2,5.405405


In [ ]:
#  ==========================================================================================================================================
#                                          1.      LONG TERM BEHAVIOUR (ROOM PREFERENCE DISTRIBUTION)
# ===========================================================================================================================================
room_type_features = (
    room_type_behavior
    .pivot(index="employee_id",
           columns="requested_room_type",
           values="percentage")
    .fillna(0)
    .reset_index()
)

room_type_features

requested_room_type,employee_id,Conference,Focus,Meeting,Training
0,E0001,0.000000,7.692308,89.230769,3.076923
1,E0002,40.000000,0.000000,60.000000,0.000000
2,E0003,0.000000,0.000000,100.000000,0.000000
3,E0004,7.142857,3.571429,82.142857,7.142857
4,E0005,0.000000,0.000000,100.000000,0.000000
...,...,...,...,...,...
995,E0996,82.258065,6.451613,6.451613,4.838710
996,E0997,11.627907,79.069767,4.651163,4.651163
997,E0998,79.545455,9.090909,9.090909,2.272727
998,E0999,4.651163,2.325581,86.046512,6.976744


In [ ]:
room_type_features[["Conference", "Focus", "Meeting", "Training"]].sum(axis=1).describe()

count    1.000000e+03
mean     1.000000e+02
std      8.005106e-15
min      1.000000e+02
25%      1.000000e+02
50%      1.000000e+02
75%      1.000000e+02
max      1.000000e+02
dtype: float64

In [ ]:
floor_behavior = (
    df.groupby(["employee_id", "requested_floor"])
      .size()
      .reset_index(name="request_count")
)

floor_behavior

,employee_id,requested_floor,request_count
0,E0001,1,9
1,E0001,2,49
2,E0001,3,3
3,E0001,4,1
4,E0001,5,3
...,...,...,...
3464,E0999,5,3
3465,E1000,1,63
3466,E1000,2,1
3467,E1000,4,1


In [ ]:
floor_behavior.groupby("employee_id").size().value_counts().sort_index()

1     88
2    148
3    227
4    281
5    256
Name: count, dtype: int64

In [ ]:
room_booking_requests = pd.read_csv(
    "../data/raw/room_booking_requests.csv",
    parse_dates=[
        "requested_start_datetime",
        "requested_end_datetime"
    ]
)

room_booking_requests.head()

,booking_id,employee_id,booking_date,requested_start_datetime,requested_end_datetime,requested_duration_minutes,requested_room_type,requested_capacity,max_time_flexibility_minutes,requested_floor
0,RB06430,E0742,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Meeting,16,60,3
1,RB10999,E0166,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Conference,4,60,5
2,RB19847,E0605,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Conference,12,0,1
3,RB22582,E0904,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Meeting,4,30,3
4,RB25100,E0378,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:00:00,60,Meeting,10,0,5


In [ ]:
#  ==========================================================================================================================================
#                                           2.     LONG TERM BEHAVIOUR (FLOOR PREFERENCE DISTRIBUTION)
# ===========================================================================================================================================
floor_behavior = (
    room_booking_requests
    .groupby(["employee_id", "requested_floor"])
    .size()
    .reset_index(name="request_count")
)

floor_behavior["percentage"] = (
    floor_behavior["request_count"]
    / floor_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

floor_behavior.head(20)

,employee_id,requested_floor,request_count,percentage
0,E0001,1,9,13.846154
1,E0001,2,49,75.384615
2,E0001,3,3,4.615385
3,E0001,4,1,1.538462
4,E0001,5,3,4.615385
5,E0002,1,10,100.000000
6,E0003,5,7,100.000000
7,E0004,1,26,92.857143
8,E0004,2,1,3.571429
9,E0004,3,1,3.571429


In [ ]:
floor_features = (
    floor_behavior
    .pivot(
        index="employee_id",
        columns="requested_floor",
        values="percentage"
    )
    .fillna(0)
    .reset_index()
)

floor_features.columns = [
    "employee_id",
    "floor_1_pct",
    "floor_2_pct",
    "floor_3_pct",
    "floor_4_pct",
    "floor_5_pct"
]

floor_features.head(10)

,employee_id,floor_1_pct,floor_2_pct,floor_3_pct,floor_4_pct,floor_5_pct
0,E0001,13.846154,75.384615,4.615385,1.538462,4.615385
1,E0002,100.000000,0.000000,0.000000,0.000000,0.000000
2,E0003,0.000000,0.000000,0.000000,0.000000,100.000000
3,E0004,92.857143,3.571429,3.571429,0.000000,0.000000
4,E0005,0.000000,0.000000,0.000000,0.000000,100.000000
5,E0006,4.878049,4.878049,78.048780,4.878049,7.317073
6,E0007,0.000000,0.000000,3.125000,96.875000,0.000000
7,E0008,0.000000,0.000000,100.000000,0.000000,0.000000
8,E0009,22.222222,77.777778,0.000000,0.000000,0.000000
9,E0010,63.333333,10.000000,16.666667,10.000000,0.000000


In [ ]:
floor_features[
    [
        "floor_1_pct",
        "floor_2_pct",
        "floor_3_pct",
        "floor_4_pct",
        "floor_5_pct"
    ]
].sum(axis=1).describe()

count    1.000000e+03
mean     1.000000e+02
std      8.055453e-15
min      1.000000e+02
25%      1.000000e+02
50%      1.000000e+02
75%      1.000000e+02
max      1.000000e+02
dtype: float64

In [ ]:
#  ==========================================================================================================================================
#                                           3.     LONG TERM BEHAVIOUR (START HOUR PREFERENCE DISTRIBUTION)
# ===========================================================================================================================================

start_hour_behavior = (
    room_booking_requests
    .assign(
        requested_start_hour=
        room_booking_requests["requested_start_datetime"].dt.hour
    )
    .groupby(["employee_id", "requested_start_hour"])
    .size()
    .reset_index(name="request_count")
)

start_hour_behavior["percentage"] = (
    start_hour_behavior["request_count"]
    / start_hour_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

start_hour_behavior.head(20)

,employee_id,requested_start_hour,request_count,percentage
0,E0001,8,9,13.846154
1,E0001,9,47,72.307692
2,E0001,10,9,13.846154
3,E0002,9,8,80.000000
4,E0002,10,2,20.000000
5,E0003,8,2,28.571429
6,E0003,9,5,71.428571
7,E0004,8,3,10.714286
8,E0004,9,20,71.428571
9,E0004,10,5,17.857143


In [ ]:
start_hour_check = room_booking_requests[
    ["employee_id", "requested_start_datetime"]
].copy()

start_hour_check["requested_start_hour"] = (
    start_hour_check["requested_start_datetime"].dt.hour
)

start_hour_check = start_hour_check.merge(
    employees[
        ["employee_id", "preferred_start_hour"]
    ],
    on="employee_id",
    how="left"
)

start_hour_check["hour_difference"] = (
    start_hour_check["requested_start_hour"]
    - start_hour_check["preferred_start_hour"]
).abs()

print("\nStart hour difference distribution:")
print(
    start_hour_check["hour_difference"]
    .value_counts()
    .sort_index()
)

print("\nStart hour preference percentages:")

print(
    "Exact preferred hour:",
    (
        start_hour_check["hour_difference"] == 0
    ).mean()
)

print(
    "Within ±1 hour:",
    (
        start_hour_check["hour_difference"] <= 1
    ).mean()
)


Start hour difference distribution:
hour_difference
0    28022
1    11978
Name: count, dtype: int64

Start hour preference percentages:
Exact preferred hour: 0.70055
Within ±1 hour: 1.0


In [ ]:
employees = pd.read_csv(
    "../data/raw/employees.csv"
)

employees.head()

,employee_id,department,work_mode,preferred_floor,preferred_zone,preferred_start_hour,preferred_room_type,typical_capacity,typical_duration_minutes,preferred_flexibility_minutes
0,E0001,Operations,Office,2,Open,9,Meeting,4,90,120
1,E0002,Finance,Remote,1,Open,9,Meeting,4,75,60
2,E0003,Sales,Remote,5,Open,9,Meeting,4,60,30
3,E0004,Operations,Hybrid,1,Open,9,Meeting,6,105,0
4,E0005,HR,Remote,5,Collaborative,9,Meeting,10,105,30


In [ ]:
#  ==========================================================================================================================================
#                                            4.    LONG TERM BEHAVIOUR (CAPACITY PREFERENCE DISTRIBUTION)
# ===========================================================================================================================================

capacity_behavior = (
    room_booking_requests
    .groupby(["employee_id", "requested_capacity"])
    .size()
    .reset_index(name="request_count")
)

capacity_behavior["percentage"] = (
    capacity_behavior["request_count"]
    / capacity_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

capacity_behavior.head(20)

,employee_id,requested_capacity,request_count,percentage
0,E0001,2,14,21.538462
1,E0001,4,32,49.230769
2,E0001,6,14,21.538462
3,E0001,8,2,3.076923
4,E0001,10,2,3.076923
5,E0001,12,1,1.538462
6,E0002,2,2,20.000000
7,E0002,4,6,60.000000
8,E0002,6,1,10.000000
9,E0002,10,1,10.000000


In [ ]:
capacity_check = room_booking_requests[
    ["employee_id", "requested_capacity"]
].merge(
    employees[
        ["employee_id", "typical_capacity"]
    ],
    on="employee_id",
    how="left"
)

capacity_check["capacity_difference"] = (
    capacity_check["requested_capacity"]
    - capacity_check["typical_capacity"]
).abs()

print("\nCapacity difference distribution:")
print(
    capacity_check["capacity_difference"]
    .value_counts()
    .sort_index()
)

print("\nCapacity preference percentages:")

print(
    "Exact typical capacity:",
    (
        capacity_check["capacity_difference"] == 0
    ).mean()
)

print(
    "Within ±2 capacity:",
    (
        capacity_check["capacity_difference"] <= 2
    ).mean()
)

print(
    "Within ±4 capacity:",
    (
        capacity_check["capacity_difference"] <= 4
    ).mean()
)


Capacity difference distribution:
capacity_difference
0     21493
2     13693
4      1537
6      1274
8       874
10      540
12      321
14      268
Name: count, dtype: int64

Capacity preference percentages:
Exact typical capacity: 0.537325
Within ±2 capacity: 0.87965
Within ±4 capacity: 0.918075


In [ ]:
duration_behavior = (
    room_booking_requests
    .groupby(["employee_id", "requested_duration_minutes"])
    .size()
    .reset_index(name="request_count")
)

duration_behavior["percentage"] = (
    duration_behavior["request_count"]
    / duration_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

duration_behavior.head(20)

,employee_id,requested_duration_minutes,request_count,percentage
0,E0001,30,10,15.384615
1,E0001,45,5,7.692308
2,E0001,60,20,30.769231
3,E0001,75,10,15.384615
4,E0001,90,13,20.000000
5,E0001,105,4,6.153846
6,E0001,120,3,4.615385
7,E0002,30,2,20.000000
8,E0002,45,2,20.000000
9,E0002,60,2,20.000000


In [ ]:
duration_check = room_booking_requests[
    ["employee_id", "requested_duration_minutes"]
].merge(
    employees[
        ["employee_id", "typical_duration_minutes"]
    ],
    on="employee_id",
    how="left"
)

duration_check["duration_difference"] = (
    duration_check["requested_duration_minutes"]
    - duration_check["typical_duration_minutes"]
).abs()

print("\nDuration difference distribution:")
print(
    duration_check["duration_difference"]
    .value_counts()
    .sort_index()
)

print("\nDuration preference percentages:")

print(
    "Exact typical duration:",
    (
        duration_check["duration_difference"] == 0
    ).mean()
)

print(
    "Within ±15 minutes:",
    (
        duration_check["duration_difference"] <= 15
    ).mean()
)

print(
    "Within ±30 minutes:",
    (
        duration_check["duration_difference"] <= 30
    ).mean()
)


Duration difference distribution:
duration_difference
0      7603
15    11455
30    10676
45     5363
60     3547
75      963
90      393
Name: count, dtype: int64

Duration preference percentages:
Exact typical duration: 0.190075
Within ±15 minutes: 0.47645
Within ±30 minutes: 0.74335


In [ ]:
flexibility_behavior = (
    room_booking_requests
    .groupby(["employee_id", "max_time_flexibility_minutes"])
    .size()
    .reset_index(name="request_count")
)

flexibility_behavior["percentage"] = (
    flexibility_behavior["request_count"]
    / flexibility_behavior.groupby("employee_id")["request_count"].transform("sum")
    * 100
)

flexibility_behavior.head(20)

,employee_id,max_time_flexibility_minutes,request_count,percentage
0,E0001,0,16,24.615385
1,E0001,15,15,23.076923
2,E0001,30,17,26.153846
3,E0001,60,15,23.076923
4,E0001,120,2,3.076923
5,E0002,0,3,30.000000
6,E0002,15,2,20.000000
7,E0002,30,3,30.000000
8,E0002,60,1,10.000000
9,E0002,120,1,10.000000


In [ ]:
flexibility_check = room_booking_requests[
    ["employee_id", "max_time_flexibility_minutes"]
].merge(
    employees[
        ["employee_id", "preferred_flexibility_minutes"]
    ],
    on="employee_id",
    how="left"
)

flexibility_check["flexibility_difference"] = (
    flexibility_check["max_time_flexibility_minutes"]
    - flexibility_check["preferred_flexibility_minutes"]
).abs()

print("\nFlexibility difference distribution:")
print(
    flexibility_check["flexibility_difference"]
    .value_counts()
    .sort_index()
)

print("\nFlexibility preference percentages:")

print(
    "Exact preferred flexibility:",
    (
        flexibility_check["flexibility_difference"] == 0
    ).mean()
)

print(
    "Within ±15 minutes:",
    (
        flexibility_check["flexibility_difference"] <= 15
    ).mean()
)

print(
    "Within ±30 minutes:",
    (
        flexibility_check["flexibility_difference"] <= 30
    ).mean()
)


Flexibility difference distribution:
flexibility_difference
0      8766
15     7324
30     9339
45     3841
60     5833
90     1877
105    1515
120    1505
Name: count, dtype: int64

Flexibility preference percentages:
Exact preferred flexibility: 0.21915
Within ±15 minutes: 0.40225
Within ±30 minutes: 0.635725
